# Lab 3 - SPAN NPU v1 (Modal App)

- Candidate: `span_npu_v1`
- Model family: Swift Parameter-free Attention Network (CVPRW 2024), adapted to same-resolution restoration with an NPU-safe attention variant.
- Expected input/output: `256x256x3`
- Primary variant: `C=28, 6 SPAB, attention=hardsigmoid, aggregation=sum, use_rep=true, use_teacher=false` (approx 143K params).
- Paper: [Wan et al., CVPRW 2024](https://openaccess.thecvf.com/content/CVPR2024W/NTIRE/papers/Wan_Swift_Parameter-free_Attention_Network_for_Efficient_Super-Resolution_CVPRW_2024_paper.pdf).

This notebook is fully self-contained per the rubric's Notebook Independence rule: model, training, REP collapse, ONNX export, calibration, operator audit, MXQ handoff, and Modal dispatch all live as inline cells below.


## 1. Setup

Imports, device resolution, project paths, config dataclass, and variant registry.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import subprocess
import sys
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass, field, fields
from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

try:
    import onnx
except Exception:
    onnx = None

try:
    import onnxruntime as ort
except Exception:
    ort = None


PROJECT_ROOT = Path(os.environ.get("LAB3_PROJECT_ROOT", Path.cwd().resolve())).resolve()
if not (PROJECT_ROOT / "Data").exists():
    # notebook launched from inside SPAN NPU v1/: walk up one level
    if (PROJECT_ROOT.parent / "Data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "Data"
RUNS_ROOT = PROJECT_ROOT / "runs"
AUTOPILOT_REPORTS = RUNS_ROOT / "autopilot_reports"
STEP2_SCRIPT = PROJECT_ROOT / "lab3_step2_onnx_to_mxq.py"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp"}

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_ROOT    = {DATA_ROOT}  exists={DATA_ROOT.exists()}")
print(f"RUNS_ROOT    = {RUNS_ROOT}")
print(f"DEVICE       = {DEVICE}")
print(f"torch        = {torch.__version__}")


In [ ]:
@dataclass(frozen=True)
class SpanConfig:
    """Primary Config for the SPAN NPU v1 port."""
    candidate_id: str = "span_default"
    channels: int = 28
    num_blocks: int = 6
    attention: str = "hardsigmoid"       # faithful | hardsigmoid | hardswish | none
    aggregation: str = "sum"             # concat | sum | sequential
    use_rep: bool = True
    use_teacher: bool = False
    teacher_weight: float = 0.0
    leaky_slope: float = 0.05
    seed: int = 12345
    # training
    batch_size: int = 16
    num_workers: int = 4
    learning_rate: float = 5e-4
    lr_schedule: str = "cosine"          # cosine | step
    budget_minutes: int = 120
    val_every_minutes: float = 5.0
    augment: bool = True
    cutout_prob: float = 0.0             # 0.0 disables
    # data
    include_df2k: bool = False
    include_lsdir: bool = False
    train_pair_cap: int | None = None
    val_pair_cap: int | None = None
    # modal
    modal_gpu: str = "L40S"
    modal_data_volume: str = "lab3-data"
    modal_runs_volume: str = "lab3-runs"

    def hash(self) -> str:
        payload = json.dumps(asdict(self), sort_keys=True).encode("utf-8")
        return hashlib.sha256(payload).hexdigest()[:12]


VALID_ATTENTION = {"faithful", "hardsigmoid", "hardswish", "none"}
VALID_AGGREGATION = {"concat", "sum", "sequential"}


def validate_cfg(cfg: SpanConfig) -> None:
    if cfg.channels <= 0:
        raise ValueError("channels must be > 0")
    if cfg.num_blocks < 1:
        raise ValueError("num_blocks must be >= 1")
    if cfg.attention not in VALID_ATTENTION:
        raise ValueError(f"attention must be one of {sorted(VALID_ATTENTION)}")
    if cfg.aggregation not in VALID_AGGREGATION:
        raise ValueError(f"aggregation must be one of {sorted(VALID_AGGREGATION)}")
    if cfg.budget_minutes <= 0:
        raise ValueError("budget_minutes must be > 0")


VARIANT_REGISTRY: dict[str, SpanConfig] = {
    "span_default":        SpanConfig(candidate_id="span_default"),
    "span_faithful":       SpanConfig(candidate_id="span_faithful", attention="faithful", aggregation="concat"),
    "span_hardswish":      SpanConfig(candidate_id="span_hardswish", attention="hardswish", aggregation="sum"),
    "span_noatt":          SpanConfig(candidate_id="span_noatt", attention="none", aggregation="sum"),
    "span_default_teacher":SpanConfig(candidate_id="span_default_teacher", use_teacher=True, teacher_weight=0.5),
    "span_large":          SpanConfig(candidate_id="span_large", channels=48),
}

PRIMARY_VARIANT = os.environ.get("LAB3_SPAN_VARIANT", "span_default")
CFG = VARIANT_REGISTRY[PRIMARY_VARIANT]

# Optional env overrides (useful for smoke tests / sweeping):
_override_budget = os.environ.get("LAB3_SPAN_BUDGET_MINUTES")
_override_batch = os.environ.get("LAB3_SPAN_BATCH_SIZE")
_override_val_cap = os.environ.get("LAB3_SPAN_VAL_PAIR_CAP")
_override_train_cap = os.environ.get("LAB3_SPAN_TRAIN_PAIR_CAP")
_override_workers = os.environ.get("LAB3_SPAN_NUM_WORKERS")
_override_gpu = os.environ.get("LAB3_SPAN_MODAL_GPU")

from dataclasses import replace as _dc_replace_main  # local alias
_overrides: dict[str, Any] = {}
if _override_budget is not None:
    _overrides["budget_minutes"] = int(_override_budget)
if _override_batch is not None:
    _overrides["batch_size"] = int(_override_batch)
if _override_val_cap is not None:
    _overrides["val_pair_cap"] = int(_override_val_cap)
if _override_train_cap is not None:
    _overrides["train_pair_cap"] = int(_override_train_cap)
if _override_workers is not None:
    _overrides["num_workers"] = int(_override_workers)
if _override_gpu is not None:
    _overrides["modal_gpu"] = _override_gpu
if _overrides:
    CFG = _dc_replace_main(CFG, **_overrides)

validate_cfg(CFG)
print(f"Active variant: {CFG.candidate_id}  hash={CFG.hash()}")
if _overrides:
    print(f"Env overrides applied: {_overrides}")
print(json.dumps(asdict(CFG), indent=2, sort_keys=True))


## 2. Data

Paired LR/HR loader for the course `Data/` layout. Same-resolution `256x256x3` pairs are already produced by the data pipeline; no crop is needed. Paper-style augmentation (random horizontal flip + random 90-degree rotation) is applied jointly to each pair. DF2K / LSDIR sources are wired behind config flags; calibration data remains strictly course-training-derived per the rubric.


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


EXPECTED_TRAIN_PAIRS = 3036
EXPECTED_VAL_PAIRS = 100


def _file_maps(directory: Path) -> tuple[dict[str, Path], list[str]]:
    if not directory.exists():
        return {}, []
    file_map: dict[str, Path] = {}
    duplicates: list[str] = []
    for path in sorted(directory.iterdir()):
        if not path.is_file() or path.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        if path.stem in file_map:
            duplicates.append(path.stem)
        else:
            file_map[path.stem] = path
    return file_map, duplicates


def audit_pair_directory(hr_dir: Path, lr_dir: Path, split_name: str) -> dict[str, Any]:
    if not hr_dir.exists():
        raise FileNotFoundError(f"Missing HR directory for {split_name}: {hr_dir}")
    if not lr_dir.exists():
        raise FileNotFoundError(f"Missing LR directory for {split_name}: {lr_dir}")

    hr_map, hr_duplicates = _file_maps(hr_dir)
    lr_map, lr_duplicates = _file_maps(lr_dir)
    shared = sorted(set(hr_map) & set(lr_map))
    hr_only = sorted(set(hr_map) - set(lr_map))
    lr_only = sorted(set(lr_map) - set(hr_map))

    size_mismatches: list[dict[str, Any]] = []
    unreadable_pairs: list[dict[str, Any]] = []
    for stem in shared:
        try:
            with Image.open(hr_map[stem]) as hr_img_raw, Image.open(lr_map[stem]) as lr_img_raw:
                hr_img = hr_img_raw.convert("RGB")
                lr_img = lr_img_raw.convert("RGB")
                if hr_img.size != lr_img.size:
                    size_mismatches.append(
                        {
                            "stem": stem,
                            "hr_size": list(hr_img.size),
                            "lr_size": list(lr_img.size),
                        }
                    )
        except Exception as exc:
            unreadable_pairs.append({"stem": stem, "error": str(exc)})

    passed = not any([hr_duplicates, lr_duplicates, hr_only, lr_only, size_mismatches, unreadable_pairs, not shared])
    return {
        "split": split_name,
        "hr_dir": str(hr_dir),
        "lr_dir": str(lr_dir),
        "hr_count": len(hr_map),
        "lr_count": len(lr_map),
        "paired_count": len(shared),
        "hr_only_count": len(hr_only),
        "lr_only_count": len(lr_only),
        "duplicate_hr_basenames": hr_duplicates,
        "duplicate_lr_basenames": lr_duplicates,
        "hr_only_examples": hr_only[:10],
        "lr_only_examples": lr_only[:10],
        "size_mismatches": size_mismatches[:10],
        "unreadable_pairs": unreadable_pairs[:10],
        "passed": passed,
    }


def run_pairing_audit(data_root: Path, enforce_expected_totals: bool = True) -> dict[str, Any]:
    train_results = []
    for idx in range(1, 5):
        train_results.append(
            audit_pair_directory(
                data_root / "HR_train" / f"HR_train{idx}",
                data_root / "LR_train" / f"LR_train{idx}",
                split_name=f"HR_train{idx}",
            )
        )
    val_result = audit_pair_directory(data_root / "HR_val", data_root / "LR_val", split_name="val")
    observed = {
        "train_pairs_observed": sum(item["paired_count"] for item in train_results),
        "val_pairs_observed": val_result["paired_count"],
    }
    expected = {
        "train_pairs_expected": EXPECTED_TRAIN_PAIRS,
        "val_pairs_expected": EXPECTED_VAL_PAIRS,
    }
    expected_total_mismatch = {}
    if enforce_expected_totals:
        if observed["train_pairs_observed"] != EXPECTED_TRAIN_PAIRS:
            expected_total_mismatch["train_pairs_observed"] = observed["train_pairs_observed"]
        if observed["val_pairs_observed"] != EXPECTED_VAL_PAIRS:
            expected_total_mismatch["val_pairs_observed"] = observed["val_pairs_observed"]
    passed = all(item["passed"] for item in train_results) and val_result["passed"] and not expected_total_mismatch
    report = {
        "expected": expected,
        "observed": observed,
        "train_splits": train_results,
        "val_split": val_result,
        "expected_total_mismatch": expected_total_mismatch,
        "enforce_expected_totals": enforce_expected_totals,
        "passed": passed,
    }
    if not passed:
        raise RuntimeError("HR/LR pairing audit failed. Inspect the report before proceeding.")
    return report


def discover_pairs(hr_root: Path, lr_root: Path) -> list[tuple[Path, Path]]:
    hr_map, _ = _file_maps(hr_root)
    lr_map, _ = _file_maps(lr_root)
    return [(lr_map[stem], hr_map[stem]) for stem in sorted(set(hr_map) & set(lr_map))]


def discover_course_pairs(data_root: Path, split: str) -> list[tuple[Path, Path]]:
    if split == "train":
        pairs: list[tuple[Path, Path]] = []
        for idx in range(1, 5):
            hr_root = data_root / "HR_train" / f"HR_train{idx}"
            lr_root = data_root / "LR_train" / f"LR_train{idx}"
            pairs.extend(discover_pairs(hr_root, lr_root))
        return pairs
    if split == "val":
        return discover_pairs(data_root / "HR_val", data_root / "LR_val")
    raise ValueError(f"unknown split: {split}")


def discover_df2k_pairs(data_root: Path) -> list[tuple[Path, Path]]:
    df2k_hr = data_root / "DF2K" / "HR"
    df2k_lr = data_root / "DF2K" / "LR"
    if not df2k_hr.exists() or not df2k_lr.exists():
        return []
    return discover_pairs(df2k_hr, df2k_lr)


def discover_lsdir_pairs(data_root: Path) -> list[tuple[Path, Path]]:
    hr_root = data_root / "LSDIR" / "HR"
    lr_root = data_root / "LSDIR" / "LR"
    if not hr_root.exists() or not lr_root.exists():
        return []
    return discover_pairs(hr_root, lr_root)


def build_training_pairs(cfg: SpanConfig, data_root: Path) -> list[tuple[Path, Path]]:
    pairs = discover_course_pairs(data_root, "train")
    if cfg.include_df2k:
        pairs.extend(discover_df2k_pairs(data_root))
    if cfg.include_lsdir:
        pairs.extend(discover_lsdir_pairs(data_root))
    if cfg.train_pair_cap is not None:
        pairs = pairs[: cfg.train_pair_cap]
    return pairs


def build_validation_pairs(cfg: SpanConfig, data_root: Path) -> list[tuple[Path, Path]]:
    pairs = discover_course_pairs(data_root, "val")
    if cfg.val_pair_cap is not None:
        pairs = pairs[: cfg.val_pair_cap]
    return pairs


def load_rgb_tensor(path: Path) -> torch.Tensor:
    with Image.open(path) as img:
        img = img.convert("RGB")
        arr = np.asarray(img, dtype=np.uint8)
    tensor = torch.from_numpy(arr).permute(2, 0, 1).contiguous().float().div_(255.0)
    return tensor


def _apply_joint_aug(lr: torch.Tensor, hr: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    if random.random() < 0.5:
        lr = torch.flip(lr, dims=[-1])
        hr = torch.flip(hr, dims=[-1])
    k = random.randint(0, 3)
    if k > 0:
        lr = torch.rot90(lr, k=k, dims=(-2, -1))
        hr = torch.rot90(hr, k=k, dims=(-2, -1))
    return lr, hr


def _maybe_cutout(lr: torch.Tensor, prob: float) -> torch.Tensor:
    if prob <= 0.0 or random.random() >= prob:
        return lr
    _, h, w = lr.shape
    size = random.randint(min(8, h // 8), max(16, h // 4))
    top = random.randint(0, max(0, h - size))
    left = random.randint(0, max(0, w - size))
    lr = lr.clone()
    lr[:, top : top + size, left : left + size] = 0.0
    return lr


class PairedRestorationDataset(Dataset):
    """Paired LR/HR dataset for Lab 3 restoration. Assumes pairs are already 256x256."""

    def __init__(
        self,
        pairs: Sequence[tuple[Path, Path]],
        *,
        augment: bool,
        cutout_prob: float = 0.0,
    ) -> None:
        self.pairs = list(pairs)
        self.augment = augment
        self.cutout_prob = cutout_prob

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        lr_path, hr_path = self.pairs[index]
        lr = load_rgb_tensor(lr_path)
        hr = load_rgb_tensor(hr_path)
        if lr.shape[-2:] != (256, 256) or hr.shape[-2:] != (256, 256):
            lr = F.interpolate(lr.unsqueeze(0), size=(256, 256), mode="bicubic", align_corners=False).squeeze(0).clamp_(0.0, 1.0)
            hr = F.interpolate(hr.unsqueeze(0), size=(256, 256), mode="bicubic", align_corners=False).squeeze(0).clamp_(0.0, 1.0)
        if self.augment:
            lr, hr = _apply_joint_aug(lr, hr)
            lr = _maybe_cutout(lr, self.cutout_prob)
        return lr, hr


def data_manifest(cfg: SpanConfig, data_root: Path) -> dict[str, Any]:
    train_pairs = build_training_pairs(cfg, data_root)
    val_pairs = build_validation_pairs(cfg, data_root)
    return {
        "data_root": str(data_root),
        "train_pairs": len(train_pairs),
        "val_pairs": len(val_pairs),
        "train_preview": [str(lr.name) for lr, _ in train_pairs[:3]],
        "val_preview": [str(lr.name) for lr, _ in val_pairs[:3]],
        "pair_limit_mode": "full" if cfg.train_pair_cap is None and cfg.val_pair_cap is None else "limited",
        "includes": {
            "course": True,
            "df2k": bool(cfg.include_df2k and len(train_pairs) > 0),
            "lsdir": bool(cfg.include_lsdir and len(train_pairs) > 0),
        },
        "io_shape": "256x256x3",
    }


# Local dry-run: confirm the course data layout is present.
if DATA_ROOT.exists():
    enforce_expected_totals = CFG.train_pair_cap is None and CFG.val_pair_cap is None
    pairing_audit = run_pairing_audit(DATA_ROOT, enforce_expected_totals=enforce_expected_totals)
    manifest = data_manifest(CFG, DATA_ROOT)
    manifest["pairing_audit"] = {
        "expected": pairing_audit["expected"],
        "observed": pairing_audit["observed"],
        "enforce_expected_totals": pairing_audit["enforce_expected_totals"],
    }
    print(json.dumps(manifest, indent=2))
    assert manifest["train_pairs"] > 0, "No training pairs found; check Data/HR_train and Data/LR_train"
    assert manifest["val_pairs"] > 0, "No validation pairs found; check Data/HR_val and Data/LR_val"
else:
    print(f"Data root {DATA_ROOT} not present locally (expected on Modal via lab3-data volume).")


## 3. Model and Training

SPAN backbone adapted to same-resolution restoration: `stem -> 6 SPAB body -> aggregation -> tail + global residual`. Attention mode, aggregation mode, and REP are configurable. The model is fully inline; no imports from repo-local helper modules.


In [ ]:
class RepConv3x3(nn.Module):
    """RepVGG-style multi-branch `Conv3x3` slot.

    During training: Conv3x3 + Conv1x1 + identity (when in_ch == out_ch).
    At export: collapse to a single Conv3x3 via `fuse()`.
    """

    def __init__(self, in_channels: int, out_channels: int, *, use_rep: bool = True) -> None:
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.use_rep = use_rep
        self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=True)
        if use_rep:
            self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, padding=0, bias=True)
            self.identity_ok = in_channels == out_channels
        else:
            self.conv1 = None
            self.identity_ok = False
        self.fused: nn.Conv2d | None = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.fused is not None:
            return self.fused(x)
        y = self.conv3(x)
        if self.conv1 is not None:
            y = y + self.conv1(x)
        if self.identity_ok:
            y = y + x
        return y

    @torch.no_grad()
    def fuse(self) -> nn.Conv2d:
        """Collapse all branches into a single equivalent Conv3x3."""
        w = self.conv3.weight.clone()
        b = self.conv3.bias.clone() if self.conv3.bias is not None else torch.zeros(self.out_channels, device=w.device)
        if self.conv1 is not None:
            w1 = self.conv1.weight
            w1_pad = F.pad(w1, [1, 1, 1, 1])
            w = w + w1_pad
            if self.conv1.bias is not None:
                b = b + self.conv1.bias
        if self.identity_ok:
            eye = torch.zeros_like(w)
            for c in range(self.out_channels):
                eye[c, c, 1, 1] = 1.0
            w = w + eye
        fused = nn.Conv2d(self.in_channels, self.out_channels, kernel_size=3, padding=1, bias=True)
        fused.weight.data.copy_(w)
        fused.bias.data.copy_(b)
        self.fused = fused
        self.conv3 = None  # type: ignore[assignment]
        self.conv1 = None
        return fused


In [ ]:
class ParameterFreeAttention(nn.Module):
    """SPAN's parameter-free attention with an NPU-friendlier mode set.

    Modes:
      - faithful:     sigma(x) = Sigmoid(x) - 0.5 (paper)
      - hardsigmoid:  sigma(x) = HardSigmoid(x) - 0.5 (primary NPU-friendlier port)
      - hardswish:    gate = HardSwish(H); output = U * gate (fused self-gate)
      - none:         gate = 1; output = U (SPAN-noatt ablation)
    """

    def __init__(self, mode: str) -> None:
        super().__init__()
        if mode not in VALID_ATTENTION:
            raise ValueError(f"unknown attention mode {mode!r}")
        self.mode = mode

    def forward(self, U: torch.Tensor, H: torch.Tensor) -> torch.Tensor:
        if self.mode == "none":
            return U
        if self.mode == "faithful":
            gate = torch.sigmoid(H) - 0.5
            return U * gate
        if self.mode == "hardsigmoid":
            gate = F.hardsigmoid(H) - 0.5
            return U * gate
        if self.mode == "hardswish":
            gate = F.hardswish(H)
            return U * gate
        raise RuntimeError(f"unreachable attention mode {self.mode!r}")


class SPAB(nn.Module):
    """Swift Parameter-free Attention Block."""

    def __init__(self, channels: int, *, slope: float, attention: str, use_rep: bool) -> None:
        super().__init__()
        self.conv1 = RepConv3x3(channels, channels, use_rep=use_rep)
        self.conv2 = RepConv3x3(channels, channels, use_rep=use_rep)
        self.conv3 = RepConv3x3(channels, channels, use_rep=use_rep)
        self.act = nn.LeakyReLU(negative_slope=slope, inplace=False)
        self.attention = ParameterFreeAttention(attention)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.act(self.conv1(x))
        h = self.act(self.conv2(h))
        h = self.conv3(h)
        U = h + x
        return self.attention(U, h)


class SPAN(nn.Module):
    """Same-resolution restoration SPAN: stem -> body -> aggregation -> tail + global residual."""

    def __init__(self, cfg: SpanConfig) -> None:
        super().__init__()
        validate_cfg(cfg)
        self.cfg = cfg
        c = cfg.channels
        self.stem = nn.Conv2d(3, c, kernel_size=3, padding=1, bias=True)
        self.stem_act = nn.LeakyReLU(negative_slope=cfg.leaky_slope, inplace=False)
        self.blocks = nn.ModuleList(
            [SPAB(c, slope=cfg.leaky_slope, attention=cfg.attention, use_rep=cfg.use_rep) for _ in range(cfg.num_blocks)]
        )
        self.agg_conv = nn.Conv2d(c, c, kernel_size=3, padding=1, bias=True)
        if cfg.aggregation == "concat":
            self.mixer = nn.Conv2d(4 * c, c, kernel_size=3, padding=1, bias=True)
        elif cfg.aggregation == "sum":
            self.mixer = nn.Conv2d(c, c, kernel_size=3, padding=1, bias=True)
        elif cfg.aggregation == "sequential":
            self.mixer = nn.Conv2d(c, c, kernel_size=3, padding=1, bias=True)
        else:
            raise ValueError(cfg.aggregation)
        self.tail = nn.Conv2d(c, 3, kernel_size=3, padding=1, bias=True)
        nn.init.zeros_(self.tail.weight)
        nn.init.zeros_(self.tail.bias)

    def _aggregate(self, outputs: list[torch.Tensor]) -> torch.Tensor:
        o_last = self.agg_conv(outputs[-1])
        if self.cfg.aggregation == "sequential":
            return o_last
        if self.cfg.aggregation == "sum":
            return outputs[0] + outputs[1] + outputs[-2] + o_last
        if self.cfg.aggregation == "concat":
            return torch.cat([outputs[0], outputs[1], outputs[-2], o_last], dim=1)
        raise RuntimeError(self.cfg.aggregation)

    def predict_delta(self, x: torch.Tensor) -> torch.Tensor:
        o = self.stem_act(self.stem(x))
        outputs = [o]
        for block in self.blocks:
            o = block(o)
            outputs.append(o)
        aggregated = self._aggregate(outputs)
        mixed = self.mixer(aggregated)
        return self.tail(mixed)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.predict_delta(x)


def count_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def fuse_rep(model: SPAN) -> SPAN:
    """Collapse every RepConv3x3 bundle into a single Conv3x3 in-place."""
    for module in model.modules():
        if isinstance(module, RepConv3x3) and module.fused is None and module.conv1 is not None:
            module.fuse()
    return model


In [ ]:
# Local sanity: forward-shape + REP parity tests (run on CPU, tiny network).
def _local_unit_tests() -> None:
    cfg_small = SpanConfig(candidate_id="unit_test", channels=8, num_blocks=2, attention="hardsigmoid", aggregation="sum", use_rep=True)
    model = SPAN(cfg_small).eval()
    x = torch.randn(1, 3, 256, 256)
    with torch.no_grad():
        y_before = model(x)
    assert y_before.shape == x.shape, f"bad output shape: {y_before.shape}"
    fused_copy = copy.deepcopy(model)
    fuse_rep(fused_copy)
    with torch.no_grad():
        y_after = fused_copy(x)
    diff = (y_before - y_after).abs().max().item()
    assert diff < 1e-4, f"REP collapse parity failed, max abs diff = {diff}"
    print(f"unit tests ok: output={tuple(y_before.shape)}, rep_parity_max_abs_diff={diff:.2e}")


_local_unit_tests()

# Primary-variant param counts for sanity (multi-branch training graph + fused export graph).
from dataclasses import replace as _dc_replace
_probe_cfg = _dc_replace(CFG, candidate_id="probe")
_probe_model = SPAN(_probe_cfg)
_probe_params_train = count_parameters(_probe_model)
_probe_fused = copy.deepcopy(_probe_model)
fuse_rep(_probe_fused)
_probe_params_fused = count_parameters(_probe_fused)
print(f"Primary variant params  (C={CFG.channels}, blocks={CFG.num_blocks}, agg={CFG.aggregation}):")
print(f"  training graph (REP on)  = {_probe_params_train:,}")
print(f"  fused export graph       = {_probe_params_fused:,}")
del _probe_model, _probe_fused


In [ ]:
def psnr_from_mse(mse: float | torch.Tensor, max_val: float = 1.0) -> float:
    mse_val = float(mse)
    if mse_val <= 1e-12:
        return 99.0
    return float(10.0 * math.log10((max_val ** 2) / mse_val))


@torch.no_grad()
def evaluate_psnr(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    mse_sum_model = 0.0
    mse_sum_input = 0.0
    total_psnr_model = 0.0
    total_psnr_input = 0.0
    total_delta = 0.0
    sample_count = 0
    count = 0
    for lr, hr in loader:
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)
        out = model(lr).clamp_(0.0, 1.0)
        pred_mse = torch.mean((out - hr) ** 2, dim=(-3, -2, -1)).clamp_min(1e-12)
        input_mse = torch.mean((lr - hr) ** 2, dim=(-3, -2, -1)).clamp_min(1e-12)
        pred_psnr = -10.0 * torch.log10(pred_mse)
        input_psnr = -10.0 * torch.log10(input_mse)
        total_psnr_model += pred_psnr.sum().item()
        total_psnr_input += input_psnr.sum().item()
        total_delta += (pred_psnr - input_psnr).sum().item()
        mse_sum_model += F.mse_loss(out, hr, reduction="sum").item()
        mse_sum_input += F.mse_loss(lr, hr, reduction="sum").item()
        sample_count += lr.size(0)
        count += hr.numel()
    mean_mse_model = mse_sum_model / max(count, 1)
    mean_mse_input = mse_sum_input / max(count, 1)
    val_psnr = total_psnr_model / max(sample_count, 1)
    input_psnr = total_psnr_input / max(sample_count, 1)
    delta_psnr = total_delta / max(sample_count, 1)
    val_psnr_global = psnr_from_mse(mean_mse_model)
    input_psnr_global = psnr_from_mse(mean_mse_input)
    return {
        "val_psnr": val_psnr,
        "input_psnr": input_psnr,
        "delta_psnr": delta_psnr,
        "val_psnr_global": val_psnr_global,
        "input_psnr_global": input_psnr_global,
        "delta_psnr_global": val_psnr_global - input_psnr_global,
        "val_loss": mean_mse_model,
    }


In [ ]:
@dataclass
class TrainingOutcome:
    best_val_psnr: float
    best_metrics: dict[str, float]
    total_steps: int
    total_epochs: int
    elapsed_seconds: float
    checkpoint_path: str
    history: list[dict[str, Any]]
    final_metrics: dict[str, float]


def _format_epoch_log(prefix: str, record: dict[str, Any]) -> str:
    parts = [prefix]
    for key, label, fmt in (
        ("epoch", "epoch", "{:>3d}"),
        ("step", "step", "{:>6d}"),
        ("elapsed_seconds", "t_s", "{:>6.0f}"),
        ("lr", "lr", "{:.2e}"),
        ("train_l1", "train_l1", "{:.5f}"),
        ("val_psnr", "val_psnr", "{:.3f}"),
        ("input_psnr", "input_psnr", "{:.3f}"),
        ("delta_psnr", "delta", "{:+.3f}"),
        ("val_loss", "val_mse", "{:.5f}"),
    ):
        if key in record and record[key] is not None:
            parts.append(f"{label}=" + fmt.format(record[key]))
    return " ".join(parts)


def train_one_budget(
    cfg: SpanConfig,
    *,
    data_root: Path,
    run_root: Path,
    teacher_forward: Callable[[torch.Tensor], torch.Tensor] | None = None,
    device: torch.device | None = None,
) -> TrainingOutcome:
    """Train SPAN for `cfg.budget_minutes` wallclock; checkpoint best val PSNR.

    Logging:
      - At start: prints a banner with variant id, config hash, dataset size, device.
      - At end of every epoch: prints an [EPOCH] line with train L1, val PSNR, delta PSNR.
      - At the cutoff validation (if mid-epoch): prints a [CUT] line with the final numbers.
      - Full per-validation records are appended to run_root/metrics.jsonl.
    """
    device = device or (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
    seed_everything(cfg.seed)

    train_pairs = build_training_pairs(cfg, data_root)
    val_pairs = build_validation_pairs(cfg, data_root)
    if not train_pairs:
        raise RuntimeError(f"no training pairs under {data_root}")
    if not val_pairs:
        raise RuntimeError(f"no validation pairs under {data_root}")

    train_ds = PairedRestorationDataset(train_pairs, augment=cfg.augment, cutout_prob=cfg.cutout_prob)
    val_ds = PairedRestorationDataset(val_pairs, augment=False)
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=device.type == "cuda",
        drop_last=True,
    )
    val_loader = DataLoader(val_ds, batch_size=max(1, cfg.batch_size), shuffle=False, num_workers=cfg.num_workers)

    steps_per_epoch = max(1, len(train_loader))
    model = SPAN(cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    total_seconds = cfg.budget_minutes * 60

    if cfg.lr_schedule == "cosine":
        scheduler_fn = lambda elapsed_sec, budget_sec: 0.5 * (1.0 + math.cos(math.pi * min(1.0, elapsed_sec / max(1.0, budget_sec)))) * (cfg.learning_rate - 5e-5) + 5e-5
    else:
        scheduler_fn = lambda elapsed_sec, budget_sec: cfg.learning_rate

    run_root.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = run_root / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)
    metrics_path = run_root / "metrics.jsonl"
    latest_status_path = run_root / "latest_status.json"

    best_val_psnr = -math.inf
    best_metrics: dict[str, float] = {}
    best_state: dict[str, torch.Tensor] | None = None
    best_ckpt = checkpoint_dir / "best.pt"
    history: list[dict[str, Any]] = []

    print("=" * 78)
    print(f"[SPAN train] candidate={cfg.candidate_id} cfg_hash={cfg.hash()}")
    print(f"  channels={cfg.channels}  num_blocks={cfg.num_blocks}  attention={cfg.attention}  aggregation={cfg.aggregation}")
    print(f"  use_rep={cfg.use_rep}  use_teacher={cfg.use_teacher}  teacher_weight={cfg.teacher_weight}")
    print(f"  train_pairs={len(train_pairs)}  val_pairs={len(val_pairs)}  batch_size={cfg.batch_size}  steps_per_epoch={steps_per_epoch}")
    print(f"  budget_minutes={cfg.budget_minutes}  optimizer=adam  lr={cfg.learning_rate}  schedule={cfg.lr_schedule}")
    print(f"  device={device}  param_count={count_parameters(model):,}")
    print("=" * 78)
    sys.stdout.flush()

    start = time.monotonic()
    step = 0
    epoch = 0
    max_steps_estimate = max(1, int(total_seconds / 0.2))

    def _run_validation(label: str, step_now: int, elapsed_now: float, epoch_now: int, train_l1_avg: float, lr_now: float) -> None:
        nonlocal best_val_psnr, best_metrics, best_state
        metrics = evaluate_psnr(model, val_loader, device)
        metrics.update({
            "label": label,
            "step": step_now,
            "epoch": epoch_now,
            "elapsed_seconds": elapsed_now,
            "lr": lr_now,
            "train_l1": train_l1_avg,
        })
        history.append(metrics)
        with metrics_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(metrics) + "\n")
        is_best = metrics["val_psnr"] > best_val_psnr
        if is_best:
            best_val_psnr = metrics["val_psnr"]
            best_metrics = metrics
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save({"state_dict": best_state, "cfg": asdict(cfg), "metrics": metrics}, best_ckpt)
        suffix = "  * new best" if is_best else ""
        print(_format_epoch_log(f"[{label}]", metrics) + suffix)
        sys.stdout.flush()
        with latest_status_path.open("w", encoding="utf-8") as handle:
            handle.write(json.dumps({"step": step_now, "elapsed_seconds": elapsed_now, "epoch": epoch_now, "best_val_psnr": best_val_psnr}) + "\n")
        model.train()

    model.train()
    cut_off = False
    while True:
        epoch += 1
        epoch_loss_accum = 0.0
        epoch_loss_count = 0
        current_lr = cfg.learning_rate
        for lr_batch, hr_batch in train_loader:
            elapsed = time.monotonic() - start
            if elapsed >= total_seconds:
                cut_off = True
                break
            lr_batch = lr_batch.to(device, non_blocking=True)
            hr_batch = hr_batch.to(device, non_blocking=True)
            current_lr = scheduler_fn(elapsed, total_seconds)
            for group in optimizer.param_groups:
                group["lr"] = current_lr
            optimizer.zero_grad(set_to_none=True)
            out = model(lr_batch)
            loss = F.l1_loss(out, hr_batch)
            if teacher_forward is not None and cfg.use_teacher and cfg.teacher_weight > 0.0:
                with torch.no_grad():
                    teacher_out = teacher_forward(lr_batch)
                loss = loss + cfg.teacher_weight * F.l1_loss(out, teacher_out)
            loss.backward()
            optimizer.step()
            epoch_loss_accum += float(loss.detach())
            epoch_loss_count += 1
            step += 1

        train_l1_avg = epoch_loss_accum / max(1, epoch_loss_count)
        if cut_off:
            _run_validation("CUT", step, time.monotonic() - start, epoch, train_l1_avg, current_lr)
            break
        _run_validation("EPOCH", step, time.monotonic() - start, epoch, train_l1_avg, current_lr)
        if time.monotonic() - start >= total_seconds:
            break

    elapsed_final = time.monotonic() - start
    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save({"state_dict": best_state, "cfg": asdict(cfg), "metrics": best_metrics}, best_ckpt)

    print("=" * 78)
    print(f"[DONE]  best_val_psnr={best_val_psnr:.3f}  epochs={epoch}  steps={step}  elapsed_s={elapsed_final:.1f}")
    print("=" * 78)
    sys.stdout.flush()

    return TrainingOutcome(
        best_val_psnr=best_val_psnr,
        best_metrics=best_metrics,
        total_steps=step,
        total_epochs=epoch,
        elapsed_seconds=elapsed_final,
        checkpoint_path=str(best_ckpt),
        history=history,
        final_metrics=history[-1] if history else {},
    )


In [ ]:
def resolve_teacher_forward(cfg: SpanConfig, project_root: Path, device: torch.device) -> Callable[[torch.Tensor], torch.Tensor] | None:
    """Lazy-load the Restormer teacher from Teacher-Student Reformer/ if enabled.

    Returns None if teacher is disabled or weights cannot be located.
    """
    if not cfg.use_teacher or cfg.teacher_weight <= 0.0:
        return None
    teacher_pkg = project_root / "Teacher-Student Reformer"
    if not teacher_pkg.exists():
        print("teacher distillation requested but Teacher-Student Reformer/ missing; skipping")
        return None
    sys.path.insert(0, str(teacher_pkg))
    try:
        from restormer_teacher.model import RestormerTeacher  # type: ignore
    except Exception as exc:
        print(f"cannot import RestormerTeacher ({exc}); skipping teacher")
        return None
    weights_candidates = list((teacher_pkg / "checkpoints").glob("*.pt")) if (teacher_pkg / "checkpoints").exists() else []
    if not weights_candidates:
        runs_teacher = project_root / "runs"
        weights_candidates = list(runs_teacher.rglob("restormer_teacher*.pt"))
    if not weights_candidates:
        print("no Restormer teacher weights found; skipping teacher")
        return None
    teacher_path = weights_candidates[0]
    print(f"loading Restormer teacher from {teacher_path}")
    teacher = RestormerTeacher().to(device).eval()
    state = torch.load(teacher_path, map_location=device)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    teacher.load_state_dict(state, strict=False)
    for p in teacher.parameters():
        p.requires_grad_(False)

    def _forward(x: torch.Tensor) -> torch.Tensor:
        return teacher(x).clamp_(0.0, 1.0)

    return _forward


In [ ]:
# ---------------- Modal app (inline, self-contained) ----------------
try:
    import modal
except Exception:
    modal = None


MODAL_APP_NAME = "lab3-span-npu-v1"
REMOTE_PROJECT_ROOT = "/root/project"
REMOTE_DATA_ROOT = "/mnt/lab3-data/Data"
REMOTE_RUNS_ROOT = "/mnt/lab3-runs"


def _modal_image() -> "modal.Image":
    return (
        modal.Image.debian_slim(python_version="3.13")
        .pip_install(
            "torch==2.10.0",
            "onnx==1.20.1",
            "onnxruntime==1.24.1",
            "numpy==2.4.2",
            "pillow==12.1.1",
        )
    )


if modal is not None:
    GLOBAL_DATA_VOLUME = modal.Volume.from_name(CFG.modal_data_volume, create_if_missing=True)
    GLOBAL_RUNS_VOLUME = modal.Volume.from_name(CFG.modal_runs_volume, create_if_missing=True)
    GLOBAL_MODAL_APP = modal.App(MODAL_APP_NAME)
    GLOBAL_MODAL_IMAGE = _modal_image()

    @GLOBAL_MODAL_APP.function(
        image=GLOBAL_MODAL_IMAGE,
        gpu=CFG.modal_gpu,
        timeout=(CFG.budget_minutes + 20) * 60,
        volumes={
            "/mnt/lab3-data": GLOBAL_DATA_VOLUME,
            "/mnt/lab3-runs": GLOBAL_RUNS_VOLUME,
        },
        name="run_span_pipeline",
    )
    def run_span_pipeline(payload: dict) -> dict:
        """Remote entrypoint: trains SPAN for `budget_minutes` on Modal, saves checkpoint + metrics."""
        import json as _json
        from dataclasses import asdict as _asdict
        from pathlib import Path as _Path
        import torch as _torch

        remote_cfg = SpanConfig(**payload["cfg"])
        started_day = payload["started_day"]
        run_name = payload["run_name"]
        run_root = _Path(REMOTE_RUNS_ROOT) / started_day / run_name
        run_root.mkdir(parents=True, exist_ok=True)
        data_root = _Path(REMOTE_DATA_ROOT)
        device = _torch.device("cuda" if _torch.cuda.is_available() else "cpu")
        enforce_expected_totals = remote_cfg.train_pair_cap is None and remote_cfg.val_pair_cap is None
        pairing_audit = run_pairing_audit(data_root, enforce_expected_totals=enforce_expected_totals)
        pair_summary = data_manifest(remote_cfg, data_root)
        teacher_forward = resolve_teacher_forward(remote_cfg, _Path(REMOTE_PROJECT_ROOT), device)
        outcome = train_one_budget(remote_cfg, data_root=data_root, run_root=run_root, teacher_forward=teacher_forward, device=device)
        summary = {
            "candidate": {"candidate_id": remote_cfg.candidate_id, "config_hash": remote_cfg.hash()},
            "config": _asdict(remote_cfg),
            "evaluation": outcome.best_metrics,
            "training": {
                "total_steps": outcome.total_steps,
                "total_epochs": outcome.total_epochs,
                "elapsed_seconds": outcome.elapsed_seconds,
                "metric_style": "mean_per_image_psnr",
            },
            "pairing_audit": pairing_audit,
            "pair_summary": pair_summary,
            "run_root": str(run_root),
            "checkpoint_path": outcome.checkpoint_path,
            "history": outcome.history,
        }
        (run_root / "summary.json").write_text(_json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        GLOBAL_RUNS_VOLUME.commit()
        return summary
else:
    GLOBAL_DATA_VOLUME = None
    GLOBAL_RUNS_VOLUME = None
    GLOBAL_MODAL_APP = None
    GLOBAL_MODAL_IMAGE = None

    def run_span_pipeline(payload: dict) -> dict:
        raise RuntimeError("modal is not installed; required for remote training")


def build_modal_app(cfg: SpanConfig) -> tuple["modal.App", Callable[[dict], dict]]:
    if modal is None or GLOBAL_MODAL_APP is None:
        raise RuntimeError("modal is not installed; required for remote training")
    return GLOBAL_MODAL_APP, run_span_pipeline


def ensure_modal_data_volume(cfg: SpanConfig, local_data_root: Path) -> dict[str, Any]:
    """Refresh the required course Data/ folders in the Modal volume before each run."""
    if modal is None or GLOBAL_DATA_VOLUME is None:
        raise RuntimeError("modal is not installed")
    if not local_data_root.exists():
        raise RuntimeError(f"local Data/ missing at {local_data_root}; cannot bootstrap volume")
    cleaned: list[str] = []
    uploaded: list[dict[str, str]] = []
    for name in ["HR_train", "LR_train", "HR_val", "LR_val"]:
        remote_dir = f"/Data/{name}"
        try:
            GLOBAL_DATA_VOLUME.remove_file(remote_dir, recursive=True)
            cleaned.append(remote_dir)
        except Exception:
            pass
    with GLOBAL_DATA_VOLUME.batch_upload(force=True) as batch:
        for name in ["HR_train", "LR_train", "HR_val", "LR_val"]:
            local_dir = local_data_root / name
            if not local_dir.exists():
                raise RuntimeError(f"missing local data directory: {local_dir}")
            batch.put_directory(str(local_dir), f"/Data/{name}")
            uploaded.append({"local": str(local_dir), "remote": f"/Data/{name}"})
    return {
        "status": "uploaded",
        "volume": cfg.modal_data_volume,
        "cleaned": cleaned,
        "uploaded": uploaded,
    }


def sync_run_from_volume(cfg: SpanConfig, started_day: str, run_name: str) -> Path:
    if modal is None:
        raise RuntimeError("modal is not installed")
    local_day_root = RUNS_ROOT / started_day
    local_day_root.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["modal", "volume", "get", cfg.modal_runs_volume, f"/{started_day}/{run_name}", str(local_day_root), "--force"],
        check=True,
    )
    return local_day_root / run_name


def launch_modal_training(cfg: SpanConfig) -> dict[str, Any]:
    validate_cfg(cfg)
    if modal is None:
        print("modal not installed; skipping remote launch")
        return {"status": "skipped", "reason": "modal-not-installed"}
    started_day = time.strftime("%Y%m%d")
    run_name = f"{cfg.candidate_id}-{cfg.hash()}-{time.strftime('%H%M%S')}"
    sync_info = ensure_modal_data_volume(cfg, DATA_ROOT)
    app, remote_fn = build_modal_app(cfg)
    payload = {"cfg": asdict(cfg), "started_day": started_day, "run_name": run_name}
    detach = os.environ.get("LAB3_SPAN_DETACH", "false").lower() == "true"
    with app.run(detach=detach):
        if detach:
            call = remote_fn.spawn(payload)
            return {
                "status": "detached",
                "call_id": call.object_id,
                "started_day": started_day,
                "run_name": run_name,
                "run_root": f"{REMOTE_RUNS_ROOT}/{started_day}/{run_name}",
                "data_volume_sync": sync_info,
            }
        summary = remote_fn.remote(payload)
    local_run_root = sync_run_from_volume(cfg, started_day, run_name)
    summary["data_volume_sync"] = sync_info
    summary["synced_local_run_root"] = str(local_run_root)
    return summary


In [ ]:
def register_autopilot_entry(cfg: SpanConfig, summary: dict[str, Any]) -> Path:
    """Append a span_npu ledger entry compatible with runs/autopilot_reports/."""
    AUTOPILOT_REPORTS.mkdir(parents=True, exist_ok=True)
    ledger_path = AUTOPILOT_REPORTS / "span_npu_ledger.jsonl"
    pair_summary = summary.get("pair_summary", {})
    entry = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
        "candidate_id": cfg.candidate_id,
        "config_hash": cfg.hash(),
        "family": "span_npu_v1",
        "attention": cfg.attention,
        "aggregation": cfg.aggregation,
        "channels": cfg.channels,
        "num_blocks": cfg.num_blocks,
        "use_rep": cfg.use_rep,
        "use_teacher": cfg.use_teacher,
        "budget_minutes": cfg.budget_minutes,
        "run_root": summary.get("synced_local_run_root") or summary.get("run_root"),
        "best_val_psnr": summary.get("evaluation", {}).get("val_psnr"),
        "delta_psnr": summary.get("evaluation", {}).get("delta_psnr"),
        "input_psnr": summary.get("evaluation", {}).get("input_psnr"),
        "total_steps": summary.get("training", {}).get("total_steps"),
        "elapsed_seconds": summary.get("training", {}).get("elapsed_seconds"),
        "comparison_signature": {
            "train_source": "course+df2k+lsdir" if (cfg.include_df2k or cfg.include_lsdir) else "course",
            "batch_size": cfg.batch_size,
            "budget_minutes": cfg.budget_minutes,
            "modal_gpu": cfg.modal_gpu,
            "data_shape": "256x256x3",
            "train_pairs": pair_summary.get("train_pairs"),
            "val_pairs": pair_summary.get("val_pairs"),
            "metric_style": summary.get("training", {}).get("metric_style", "mean_per_image_psnr"),
        },
    }
    with ledger_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(entry, sort_keys=True) + "\n")
    return ledger_path


In [ ]:
# Single-variant dispatch. Flip LAB3_SPAN_LAUNCH=true to actually submit the primary variant.
LAUNCH_MODAL = os.environ.get("LAB3_SPAN_LAUNCH", "false").lower() == "true"

if LAUNCH_MODAL:
    summary = launch_modal_training(CFG)
    ledger_path = register_autopilot_entry(CFG, summary)
    print(f"ledger -> {ledger_path}")
    print(json.dumps(summary.get("evaluation", {}), indent=2, sort_keys=True))
else:
    print("LAUNCH_MODAL=false; single-variant dispatch skipped. Set env LAB3_SPAN_LAUNCH=true to run on Modal.")
    summary = {}


In [ ]:
def launch_modal_sweep(variant_ids: Sequence[str], *, parallel: bool = True) -> dict[str, Any]:
    """Dispatch a multi-variant Modal sweep. Each variant trains for cfg.budget_minutes wallclock.

    When parallel=True, variants are spawned concurrently on Modal (one worker per variant) and
    run in parallel; when False they run serially within the same app.run() context.
    """
    if modal is None:
        raise RuntimeError("modal is not installed")
    cfgs = [VARIANT_REGISTRY[v] for v in variant_ids]
    for cfg in cfgs:
        validate_cfg(cfg)
    if not cfgs:
        raise ValueError("no variants to launch")
    # Use the first cfg's gpu / volume names as the app-level defaults; all variants share volumes.
    head = cfgs[0]
    ensure_modal_data_volume(head, DATA_ROOT)
    started_day = time.strftime("%Y%m%d")
    app, remote_fn = build_modal_app(head)
    payloads: list[dict[str, Any]] = []
    for cfg in cfgs:
        payloads.append({
            "cfg": asdict(cfg),
            "started_day": started_day,
            "run_name": f"{cfg.candidate_id}-{cfg.hash()}-{time.strftime('%H%M%S')}",
        })
    sweep_results: dict[str, Any] = {"started_day": started_day, "runs": {}}
    with app.run():
        if parallel:
            calls = [remote_fn.spawn(payload) for payload in payloads]
            for cfg, payload, call in zip(cfgs, payloads, calls):
                sweep_results["runs"][cfg.candidate_id] = {"call_id": call.object_id, "payload": payload}
            # Block until all finish.
            for cfg, call in zip(cfgs, calls):
                result = call.get()
                entry = sweep_results["runs"][cfg.candidate_id]
                entry["summary"] = result
        else:
            for cfg, payload in zip(cfgs, payloads):
                result = remote_fn.remote(payload)
                sweep_results["runs"][cfg.candidate_id] = {"summary": result, "payload": payload}
    # Post-sweep: sync each run, register ledger entry.
    for cfg, payload_entry in zip(cfgs, payloads):
        run_name = payload_entry["run_name"]
        try:
            local_run_root = sync_run_from_volume(cfg, started_day, run_name)
        except Exception as exc:
            print(f"sync failed for {cfg.candidate_id}: {exc}")
            continue
        summary_rec = sweep_results["runs"][cfg.candidate_id].get("summary", {})
        summary_rec["synced_local_run_root"] = str(local_run_root)
        sweep_results["runs"][cfg.candidate_id]["summary"] = summary_rec
        register_autopilot_entry(cfg, summary_rec)
    return sweep_results


LAUNCH_SWEEP = os.environ.get("LAB3_SPAN_SWEEP", "false").lower() == "true"
SWEEP_VARIANTS = os.environ.get("LAB3_SPAN_SWEEP_VARIANTS", "span_default,span_faithful,span_hardswish,span_noatt").split(",")
SWEEP_PARALLEL = os.environ.get("LAB3_SPAN_SWEEP_PARALLEL", "true").lower() == "true"

if LAUNCH_SWEEP:
    sweep_results = launch_modal_sweep(SWEEP_VARIANTS, parallel=SWEEP_PARALLEL)
    print(json.dumps({k: v.get("summary", {}).get("evaluation", {}) for k, v in sweep_results["runs"].items()}, indent=2, sort_keys=True))
else:
    print("LAB3_SPAN_SWEEP=false; multi-variant sweep skipped.")
    print(f"  variants on deck: {SWEEP_VARIANTS}")
    print("  set LAB3_SPAN_SWEEP=true to launch the full sweep (parallel Modal workers, ~2h wallclock per variant).")
    sweep_results = {}


## 4. Validation

After the Modal run syncs back to `runs/<YYYYMMDD>/<run_name>/`, read the best-checkpoint metrics here. Reports mean validation PSNR on `Data/HR_val` and improvement over the identity-LR baseline.


In [ ]:
def load_run_summary(run_root: Path) -> dict[str, Any]:
    summary_path = run_root / "summary.json"
    if not summary_path.exists():
        raise FileNotFoundError(summary_path)
    return json.loads(summary_path.read_text(encoding="utf-8"))


def print_validation_summary(summary: dict[str, Any]) -> None:
    ev = summary.get("evaluation", {})
    print("Validation summary")
    print(f"  best val PSNR   : {ev.get('val_psnr', '-'):>7}")
    print(f"  input   PSNR    : {ev.get('input_psnr', '-'):>7}")
    print(f"  delta   PSNR    : {ev.get('delta_psnr', '-'):>7}")
    print(f"  val loss (mse)  : {ev.get('val_loss', '-')}")
    print(f"  steps trained   : {summary.get('training', {}).get('total_steps', '-')}")
    print(f"  elapsed seconds : {summary.get('training', {}).get('elapsed_seconds', '-')}")


if summary:
    print_validation_summary(summary)
else:
    print("No summary loaded yet; once the Modal run completes, re-run this cell or call load_run_summary(Path('runs/.../<run_name>')).")


## 5. Export and Submission Artifacts

REP collapse, ONNX export with parity check, training-derived calibration export, operator-risk audit, and MXQ handoff. All export cells operate on the best checkpoint synced from Modal; set `EXPORT_RUN_ROOT` below if operating on a specific run.


In [ ]:
EXPORT_RUN_ROOT_RAW = os.environ.get("LAB3_SPAN_RUN_ROOT", summary.get("synced_local_run_root", "") if summary else "")
EXPORT_RUN_ROOT = Path(EXPORT_RUN_ROOT_RAW) if EXPORT_RUN_ROOT_RAW else None
DEFER_LOCAL_EXPORT = bool(summary and summary.get("status") == "detached" and not summary.get("synced_local_run_root"))

def _pick_latest_run_root() -> Path | None:
    if EXPORT_RUN_ROOT and EXPORT_RUN_ROOT.exists():
        return EXPORT_RUN_ROOT
    if DEFER_LOCAL_EXPORT:
        return None
    if not RUNS_ROOT.exists():
        return None
    candidates = []
    for day_dir in RUNS_ROOT.iterdir():
        if not day_dir.is_dir() or day_dir.name == "autopilot_reports":
            continue
        for run_dir in day_dir.iterdir():
            if run_dir.is_dir() and run_dir.name.startswith("span_"):
                candidates.append(run_dir)
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


RUN_ROOT = _pick_latest_run_root()
print(f"RUN_ROOT for export = {RUN_ROOT}")


def _load_checkpoint(run_root: Path) -> tuple[SpanConfig, nn.Module]:
    ckpt_path = run_root / "checkpoints" / "best.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"no checkpoint at {ckpt_path}")
    payload = torch.load(ckpt_path, map_location="cpu")
    cfg_dict = payload["cfg"]
    cfg_dict.pop("modal_gpu", None)  # drop runtime-only fields if outdated
    # Filter to known SpanConfig fields for forward-compat.
    known = {f.name for f in fields(SpanConfig)}
    cfg_dict = {k: v for k, v in cfg_dict.items() if k in known}
    export_cfg = SpanConfig(**cfg_dict)
    model = SPAN(export_cfg)
    model.load_state_dict(payload["state_dict"])
    model.eval()
    return export_cfg, model


def export_onnx(run_root: Path) -> dict[str, Any]:
    export_cfg, model = _load_checkpoint(run_root)
    fuse_rep(model)  # collapse RepConv3x3 -> Conv3x3 before export
    exports_dir = run_root / "exports"
    exports_dir.mkdir(parents=True, exist_ok=True)
    onnx_path = exports_dir / "span_npu_v1.onnx"
    dummy = torch.randn(1, 3, 256, 256)
    torch.onnx.export(
        model,
        dummy,
        str(onnx_path),
        input_names=["input"],
        output_names=["output"],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes=None,
    )

    checker_ok = False
    parity = {"torch_vs_ort_max_abs": None, "torch_vs_ort_mean_abs": None}
    if onnx is not None:
        model_onnx = onnx.load(str(onnx_path))
        onnx.checker.check_model(model_onnx)
        checker_ok = True
    if ort is not None:
        sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        with torch.no_grad():
            torch_out = model(dummy).numpy()
        ort_out = sess.run(None, {"input": dummy.numpy()})[0]
        diff = np.abs(torch_out - ort_out)
        parity = {"torch_vs_ort_max_abs": float(diff.max()), "torch_vs_ort_mean_abs": float(diff.mean())}

    record = {
        "onnx_path": str(onnx_path),
        "checker_ok": checker_ok,
        **parity,
        "cfg_hash": export_cfg.hash(),
        "params": count_parameters(model),
    }
    (exports_dir / "onnx_export_report.json").write_text(json.dumps(record, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return record


if RUN_ROOT is not None:
    onnx_report = export_onnx(RUN_ROOT)
    print(json.dumps(onnx_report, indent=2, sort_keys=True))
else:
    print("no run root available; skip ONNX export")
    onnx_report = None


In [ ]:
def export_calibration(run_root: Path, cfg: SpanConfig, max_samples: int = 64) -> dict[str, Any]:
    """Export training-derived calibration tensors to exports/calibration/."""
    exports_dir = run_root / "exports" / "calibration"
    exports_dir.mkdir(parents=True, exist_ok=True)
    train_pairs = build_training_pairs(cfg, DATA_ROOT)
    if not train_pairs:
        raise RuntimeError("no training pairs available for calibration (ensure Data/ or lab3-data is mounted locally)")
    rng = random.Random(cfg.seed)
    sampled = rng.sample(train_pairs, k=min(max_samples, len(train_pairs)))
    manifest_entries: list[dict[str, Any]] = []
    for idx, (lr_path, _hr_path) in enumerate(sampled):
        tensor = load_rgb_tensor(lr_path).unsqueeze(0).numpy().astype(np.float32)
        out_path = exports_dir / f"calib_{idx:04d}.npy"
        np.save(out_path, tensor)
        manifest_entries.append({"file": out_path.name, "source_lr": str(lr_path)})
    manifest = {
        "count": len(manifest_entries),
        "derived_from_training": True,
        "dtype": "float32",
        "shape": [1, 3, 256, 256],
        "entries": manifest_entries,
    }
    manifest_path = exports_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {"manifest_path": str(manifest_path), "calibration_dir": str(exports_dir), "count": len(manifest_entries)}


if RUN_ROOT is not None and DATA_ROOT.exists():
    # calibration requires local Data/ access; skip silently when only remote
    try:
        calib_report = export_calibration(RUN_ROOT, CFG)
        print(json.dumps(calib_report, indent=2, sort_keys=True))
    except RuntimeError as exc:
        calib_report = None
        print(f"calibration skipped: {exc}")
else:
    calib_report = None
    print("skipping calibration export (no RUN_ROOT or DATA_ROOT not locally mounted)")


In [ ]:
# Operator-risk audit: compare the ONNX graph ops against the MLA-100 supported/fallback table.
MLA_PREFERRED = {
    "Conv", "LeakyRelu", "Prelu", "HardSigmoid", "HardSwish", "DepthToSpace",
}
MLA_FALLBACK = {
    "Add", "Sub", "Mul", "Sigmoid", "Concat", "Reshape", "Transpose", "Gather",
    "Cast", "Clip", "Div", "Relu", "Tanh", "Softmax", "ReduceMean",
}
MLA_UNSUPPORTED = {
    "Abs", "Ceil", "Conv3d", "Floor", "Log", "Sqrt", "Sin", "Tril",
    "GridSample", "NonMaxSuppression", "NonZero", "Or", "Xor",
}


def operator_audit(onnx_path: Path) -> dict[str, Any]:
    if onnx is None:
        return {"status": "onnx-not-available"}
    model_onnx = onnx.load(str(onnx_path))
    op_counter: dict[str, int] = {}
    for node in model_onnx.graph.node:
        op_counter[node.op_type] = op_counter.get(node.op_type, 0) + 1
    risks = {
        "preferred": {op: op_counter[op] for op in sorted(op_counter) if op in MLA_PREFERRED},
        "fallback": {op: op_counter[op] for op in sorted(op_counter) if op in MLA_FALLBACK},
        "unsupported": {op: op_counter[op] for op in sorted(op_counter) if op in MLA_UNSUPPORTED},
        "unknown": {op: op_counter[op] for op in sorted(op_counter) if op not in MLA_PREFERRED | MLA_FALLBACK | MLA_UNSUPPORTED},
    }
    total_fallback = sum(risks["fallback"].values())
    total_unsupported = sum(risks["unsupported"].values())
    summary = {
        "all_ops": op_counter,
        "risks": risks,
        "total_preferred": sum(risks["preferred"].values()),
        "total_fallback": total_fallback,
        "total_unsupported": total_unsupported,
        "npu_ready": total_unsupported == 0,
    }
    return summary


if RUN_ROOT is not None and onnx_report is not None:
    audit = operator_audit(Path(onnx_report["onnx_path"]))
    audit_path = RUN_ROOT / "exports" / "operator_audit.json"
    audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps(audit, indent=2, sort_keys=True))
else:
    audit = None
    print("skipping operator audit (no ONNX export)")


In [ ]:
def mxq_handoff(run_root: Path, onnx_path: Path, calib_dir: Path | None) -> dict[str, Any]:
    """Reference the repo-level `lab3_step2_onnx_to_mxq.py` as the conversion script.

    Per the rubric, a companion script is allowed if the notebook either generates it directly
    or contains the exact handoff path. We reference the existing script and record an explicit
    handoff payload so MXQ compilation can be executed non-interactively.
    """
    if not STEP2_SCRIPT.exists():
        raise FileNotFoundError(f"MXQ conversion script not found at {STEP2_SCRIPT}")
    mxq_path = run_root / "exports" / "span_npu_v1.mxq"
    payload = {
        "status": "handoff_recorded",
        "onnx_path": str(onnx_path),
        "mxq_target_path": str(mxq_path),
        "calibration_dir": str(calib_dir) if calib_dir else None,
        "conversion_script": str(STEP2_SCRIPT),
        "command": (
            f"python '{STEP2_SCRIPT}' --onnx '{onnx_path}' --output '{mxq_path}'"
            + (f" --calibration-dir '{calib_dir}'" if calib_dir else "")
        ),
    }
    handoff_path = run_root / "exports" / "mxq_handoff.json"
    handoff_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return payload


if RUN_ROOT is not None and onnx_report is not None:
    handoff = mxq_handoff(
        RUN_ROOT,
        Path(onnx_report["onnx_path"]),
        Path(calib_report["calibration_dir"]) if calib_report else None,
    )
    print(json.dumps(handoff, indent=2, sort_keys=True))
else:
    handoff = None
    print("skipping MXQ handoff (no ONNX export)")


In [ ]:
def measure_npu_latency(
    mxq_path: Path,
    *,
    variant: str,
    candidate_id: str,
    batch_size: int = 1,
    warmup_iters: int = 20,
    timed_iters: int = 200,
    save_path: Path | None = None,
) -> dict[str, Any]:
    """Run NPU latency measurement against an MLA-100 compiled `.mxq` artifact.

    This cell is a hardware-only template. It imports the Mobilint MLA-100 runtime
    (`maccel` / `mobilint`) lazily. On a machine WITHOUT an NPU the import fails
    cleanly and a stub record is emitted so the aggregator can still format a row.
    """
    import statistics
    record: dict[str, Any] = {
        "variant": variant,
        "candidate_id": candidate_id,
        "mxq_path": str(mxq_path),
        "batch_size": batch_size,
        "device": "mobilint-mla-100",
    }
    try:
        import maccel  # type: ignore
    except Exception as exc:
        record.update({
            "status": "npu-runtime-missing",
            "note": f"NPU runtime import failed ({exc}); run this cell on a Mobilint-equipped host.",
        })
        if save_path is not None:
            save_path.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        return record

    if not mxq_path.exists():
        record.update({"status": "mxq-missing", "note": f"MXQ artifact not found at {mxq_path}; run the MXQ handoff first."})
        if save_path is not None:
            save_path.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        return record

    model = maccel.Model(str(mxq_path))  # type: ignore[attr-defined]
    dummy = np.random.rand(batch_size, 3, 256, 256).astype(np.float32)
    for _ in range(warmup_iters):
        model.infer(dummy)
    samples: list[float] = []
    for _ in range(timed_iters):
        t0 = time.perf_counter()
        model.infer(dummy)
        samples.append((time.perf_counter() - t0) * 1000.0)
    samples.sort()
    mean_ms = statistics.fmean(samples)
    p50_ms = samples[len(samples) // 2]
    p90_ms = samples[int(len(samples) * 0.9)]
    record.update({
        "status": "ok",
        "mean_latency_ms": mean_ms,
        "p50_ms": p50_ms,
        "p90_ms": p90_ms,
        "throughput_fps": 1000.0 / max(mean_ms, 1e-6),
        "warmup_iters": warmup_iters,
        "timed_iters": timed_iters,
    })
    if save_path is not None:
        save_path.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return record


# Measure latency for the current run's MXQ artifact (no-op when Mobilint runtime or MXQ file is missing).
if RUN_ROOT is not None and onnx_report is not None:
    _mxq_candidate = Path(onnx_report["onnx_path"]).with_suffix(".mxq")
    _latency_path = RUN_ROOT / "exports" / "npu_latency.json"
    _latency_record = measure_npu_latency(
        _mxq_candidate,
        variant=CFG.candidate_id,
        candidate_id=CFG.candidate_id,
        save_path=_latency_path,
    )
    print(json.dumps(_latency_record, indent=2, sort_keys=True))
else:
    print("skipping NPU latency harness (no RUN_ROOT)")


def aggregate_npu_latency(variants: Sequence[str] = ("span_default", "span_faithful", "span_hardswish", "span_noatt")) -> dict[str, Any]:
    """Read per-variant NPU latency reports saved to runs/<day>/<run>/exports/npu_latency.json.

    The latency file is produced by separate Mobilint MLA-100 tooling outside Modal (NPU hardware is
    not on Modal). Expected schema:
      {"variant": str, "candidate_id": str, "mean_latency_ms": float, "p50_ms": float, "p90_ms": float,
       "throughput_fps": float, "batch_size": int, "device": str, "notes": str}

    Writes `runs/autopilot_reports/span_npu_latency.json` keyed by variant id.
    """
    AUTOPILOT_REPORTS.mkdir(parents=True, exist_ok=True)
    aggregate: dict[str, Any] = {}
    for variant in variants:
        latest_report: dict[str, Any] | None = None
        latest_mtime = -1.0
        if RUNS_ROOT.exists():
            for day_dir in RUNS_ROOT.iterdir():
                if not day_dir.is_dir() or day_dir.name == "autopilot_reports":
                    continue
                for run_dir in day_dir.iterdir():
                    if not run_dir.is_dir() or not run_dir.name.startswith(f"{variant}-"):
                        continue
                    latency_path = run_dir / "exports" / "npu_latency.json"
                    if latency_path.exists() and latency_path.stat().st_mtime > latest_mtime:
                        latest_mtime = latency_path.stat().st_mtime
                        latest_report = json.loads(latency_path.read_text(encoding="utf-8"))
                        latest_report["run_root"] = str(run_dir)
        aggregate[variant] = latest_report or {"status": "missing", "note": "no npu_latency.json found; run MLA-100 latency harness and save to exports/npu_latency.json"}
    out_path = AUTOPILOT_REPORTS / "span_npu_latency.json"
    out_path.write_text(json.dumps(aggregate, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {"path": str(out_path), "variants": aggregate}


npu_latency_summary = aggregate_npu_latency()
print(json.dumps(npu_latency_summary, indent=2, sort_keys=True))


In [ ]:
def final_artifact_summary(run_root: Path | None) -> dict[str, Any]:
    if run_root is None:
        return {"status": "no-run"}
    paths = {
        "run_root": str(run_root),
        "pth": str(run_root / "checkpoints" / "best.pt"),
        "onnx": str(run_root / "exports" / "span_npu_v1.onnx"),
        "mxq": str(run_root / "exports" / "span_npu_v1.mxq"),
        "calibration_manifest": str(run_root / "exports" / "calibration" / "manifest.json"),
        "operator_audit": str(run_root / "exports" / "operator_audit.json"),
        "mxq_handoff": str(run_root / "exports" / "mxq_handoff.json"),
        "summary": str(run_root / "summary.json"),
    }
    exist_flags = {key: Path(value).exists() for key, value in paths.items() if key != "run_root"}
    return {"paths": paths, "exists": exist_flags}


final = final_artifact_summary(RUN_ROOT)
print(json.dumps(final, indent=2, sort_keys=True))
